# 📘 W6 Python Lab — Hybrid RAG + 출처 인용 Q&A

**n8n W6 강의의 Python 버전.** PDF 자동 임베딩 + Cohere Reranker + 출처 인용 Q&A 완성.

## 학습 목표
- `pypdf`로 PDF에서 페이지별 텍스트 추출
- `langchain-text-splitters`로 청킹 (1000자, 200 오버랩)
- 메타데이터(파일명·페이지) 자동 부여
- Cohere Rerank API로 검색 결과 재정렬 (정확도 +40%)
- Claude가 Top-3 청크로 답변 + 출처 인용

## 🛠 사전 준비

```bash
pip install pypdf langchain-text-splitters cohere anthropic supabase openai python-dotenv
```

`.env` 추가:
```
COHERE_API_KEY=여러분의_Cohere_Trial_키
```

`pdfs/` 폴더에 FOMC 의사록이나 증권사 리포트 PDF 2~3개 넣기.

## 1. 환경 로드 + 클라이언트 초기화

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from supabase import create_client
from openai import OpenAI
import cohere
from anthropic import Anthropic

load_dotenv()

supabase = create_client(os.getenv('SUPABASE_URL'), os.getenv('SUPABASE_KEY'))
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
cohere_client = cohere.Client(os.getenv('COHERE_API_KEY'))
claude = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

print('✓ 모든 클라이언트 준비 완료')

## 2. PDF 텍스트 추출 (페이지별)

n8n의 Extract From PDF 노드 = `pypdf.PdfReader`.

In [ ]:
def extract_pdf_pages(pdf_path: str) -> list[dict]:
    """PDF를 페이지별로 추출. 각 페이지는 dict."""
    reader = PdfReader(pdf_path)
    filename = Path(pdf_path).name
    
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if len(text.strip()) < 50:
            continue  # 빈 페이지 스킵
        pages.append({
            'content': text.strip(),
            'source': filename,
            'page': i + 1
        })
    
    print(f'✓ {filename}: {len(pages)}개 유효 페이지')
    return pages

# 테스트 — pdfs 폴더의 첫 번째 PDF
pdf_files = list(Path('pdfs').glob('*.pdf')) if Path('pdfs').exists() else []
if pdf_files:
    sample_pages = extract_pdf_pages(str(pdf_files[0]))
    print(f'\n첫 페이지 미리보기:')
    print(sample_pages[0]['content'][:300] + '...')
else:
    print('⚠ pdfs/ 폴더에 PDF 파일이 없습니다. FOMC 의사록이나 증권사 리포트를 넣으세요.')

## 3. 청킹 — 페이지를 1000자 단위로 분할

페이지가 너무 크면 임베딩 정확도 저하. Recursive Character로 자연스럽게 분할.

In [ ]:
def chunk_pages(pages: list[dict], chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """페이지 텍스트를 청크로 분할. 메타데이터 유지."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    
    chunks = []
    for page in pages:
        sub_texts = splitter.split_text(page['content'])
        for j, text in enumerate(sub_texts):
            chunks.append({
                'content': text,
                'metadata': {
                    'source': page['source'],
                    'page': page['page'],
                    'chunk_in_page': j + 1,
                    'uploaded_at': datetime.now().isoformat()
                }
            })
    
    print(f'✓ {len(pages)}개 페이지 → {len(chunks)}개 청크 (평균 {len(chunks)/len(pages):.1f}개/페이지)')
    return chunks

if pdf_files:
    chunks = chunk_pages(sample_pages)
    print(f'\n첫 청크 미리보기:')
    print(f'  길이: {len(chunks[0]["content"])} 문자')
    print(f'  메타: {chunks[0]["metadata"]}')

## 4. 배치 임베딩 + Supabase 저장

W5보다 양이 많으므로 **배치 처리**(한 번에 100개씩).

In [ ]:
def embed_and_store(chunks: list[dict], batch_size: int = 100):
    """청크들을 임베딩 후 Supabase에 일괄 저장."""
    total = len(chunks)
    saved = 0
    
    for i in range(0, total, batch_size):
        batch = chunks[i:i+batch_size]
        texts = [c['content'] for c in batch]
        
        # 한 번의 API 호출로 여러 임베딩
        response = openai_client.embeddings.create(
            model='text-embedding-3-small',
            input=texts
        )
        
        # Supabase에 저장할 행 준비
        rows = [{
            'content': c['content'],
            'metadata': c['metadata'],
            'embedding': response.data[j].embedding
        } for j, c in enumerate(batch)]
        
        supabase.table('documents').insert(rows).execute()
        saved += len(batch)
        print(f'  배치 {i//batch_size + 1}: {saved}/{total} 저장됨')
    
    print(f'✓ 총 {saved}개 청크 저장 완료')

# 전체 PDF 폴더 처리
def ingest_all_pdfs(pdf_dir: str = 'pdfs'):
    """pdfs/ 폴더의 모든 PDF를 자동 임베딩."""
    all_chunks = []
    for pdf_path in Path(pdf_dir).glob('*.pdf'):
        pages = extract_pdf_pages(str(pdf_path))
        chunks = chunk_pages(pages)
        all_chunks.extend(chunks)
    
    if all_chunks:
        embed_and_store(all_chunks)
    return len(all_chunks)

# 실행
if pdf_files:
    ingest_all_pdfs()

## 5. Vector 검색 (Top 20 후보 수집)

Hybrid RAG 1단계: 넓은 그물로 20개 후보 수집.

In [ ]:
def vector_search(query: str, top_k: int = 20) -> list[dict]:
    """Vector 유사도로 Top-K 후보 수집."""
    query_vec = openai_client.embeddings.create(
        model='text-embedding-3-small',
        input=query
    ).data[0].embedding
    
    result = supabase.rpc('match_documents', {
        'query_embedding': query_vec,
        'match_count': top_k
    }).execute()
    
    return result.data

# 테스트
candidates = vector_search('FOMC 점도표 금리 전망', top_k=20)
print(f'✓ Vector 1차 검색: {len(candidates)}개 후보')
print(f'\n상위 3개 (Vector 점수):')
for i, doc in enumerate(candidates[:3]):
    print(f'{i+1}. [sim {doc["similarity"]:.3f}] {doc["content"][:80]}...')
    print(f'   출처: {doc["metadata"]["source"]} p.{doc["metadata"]["page"]}')

## 6. Cohere Rerank (Top 3로 정제)

Hybrid RAG 2단계: 20개를 Cohere가 정밀 재정렬 → Top 3만 선별.

In [ ]:
def rerank_candidates(query: str, candidates: list[dict], top_n: int = 3) -> list[dict]:
    """Cohere Reranker로 후보 재정렬 + Top-N 반환."""
    documents = [c['content'] for c in candidates]
    
    response = cohere_client.rerank(
        model='rerank-v3.5',
        query=query,
        documents=documents,
        top_n=top_n
    )
    
    # Reranker가 반환한 순서대로 원본 후보 재구성
    reranked = []
    for r in response.results:
        original = candidates[r.index]
        reranked.append({
            **original,
            'rerank_score': r.relevance_score
        })
    
    return reranked

# 테스트
top3 = rerank_candidates('FOMC 점도표 금리 전망', candidates, top_n=3)
print(f'✓ Cohere Rerank 완료\n')
print('최종 Top 3:')
for i, doc in enumerate(top3):
    print(f'{i+1}. [rerank {doc["rerank_score"]:.3f}] {doc["content"][:100]}...')
    print(f'   출처: {doc["metadata"]["source"]} p.{doc["metadata"]["page"]}\n')

## 7. Claude로 출처 인용 답변 생성

Hybrid RAG 3단계: Top 3 청크를 Claude에 컨텍스트로 전달 → 답변 + 출처.

In [ ]:
def answer_with_citations(query: str, top_chunks: list[dict]) -> str:
    """Claude가 Top-K 청크로 답변 + 출처 인용."""
    
    # 컨텍스트 구성 — 각 청크에 번호 부여
    context_text = '\n\n'.join([
        f'[출처 {i+1}] {c["metadata"]["source"]} (p.{c["metadata"]["page"]}):\n{c["content"]}'
        for i, c in enumerate(top_chunks)
    ])
    
    system = """당신은 투자 리서치 어시스턴트입니다. 제공된 출처 문서만 근거로 답합니다.

규칙:
1. 답변에 사용한 출처를 반드시 (파일명, p.페이지) 형식으로 인용
2. 출처에 정보가 없으면 '제공된 출처에 해당 정보가 없습니다'라고 정직하게 답변
3. 절대 추측 금지
4. 답변은 2~3문장으로 간결하게
"""
    
    user = f"""질문: {query}

참고 자료:
{context_text}

위 자료만 근거로 답하세요."""
    
    message = claude.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=800,
        system=system,
        messages=[{'role': 'user', 'content': user}]
    )
    
    return message.content[0].text

# 테스트
query = 'FOMC 3월 회의에서 2026년 금리 중간값은?'
answer = answer_with_citations(query, top3)
print(f'Q. {query}\n')
print(f'A. {answer}')

## 8. 전체 RAG 파이프라인 — 한 번에

Vector 검색 → Rerank → 답변 생성을 하나의 함수로.

In [ ]:
def ask(query: str, verbose: bool = True) -> dict:
    """완전한 Hybrid RAG Q&A 파이프라인.
    
    Vector 20개 → Rerank 3개 → Claude 답변 + 출처
    """
    # 1. Vector 1차 검색
    candidates = vector_search(query, top_k=20)
    if verbose: print(f'  ✓ Vector: {len(candidates)}개 후보')
    
    # 2. Cohere Rerank
    top3 = rerank_candidates(query, candidates, top_n=3)
    if verbose: print(f'  ✓ Rerank: Top 3 선별')
    
    # 3. 답변 생성
    answer = answer_with_citations(query, top3)
    
    # 4. 구조화된 응답
    sources = [{
        'file': c['metadata']['source'],
        'page': c['metadata']['page'],
        'rerank_score': round(c['rerank_score'], 3)
    } for c in top3]
    
    return {
        'query': query,
        'answer': answer,
        'sources': sources
    }

# 여러 질문 연속 테스트
questions = [
    'FOMC 점도표 2026년 중간값은?',
    '최근 연준의 금리 정책 기조는?',
    '인플레이션 둔화 관련 언급은?'
]

for q in questions:
    print(f'\n{"=" * 60}')
    result = ask(q)
    print(f'Q. {result["query"]}\n')
    print(f'A. {result["answer"]}\n')
    print('📚 참고:')
    for s in result['sources']:
        print(f'   • {s["file"]} p.{s["page"]} (score {s["rerank_score"]})')

## 9. Vector만 vs Rerank 성능 비교

Reranker가 실제로 얼마나 차이를 만드는지 측정.

In [ ]:
def compare_search_quality(query: str):
    """Vector만 vs Vector+Rerank Top 3 비교."""
    # Vector만 (Top 3)
    vector_only = vector_search(query, top_k=3)
    
    # Vector 20 + Rerank Top 3
    candidates = vector_search(query, top_k=20)
    reranked = rerank_candidates(query, candidates, top_n=3)
    
    print(f'\nQ: {query}\n')
    print('=== Vector만 Top 3 ===')
    for i, doc in enumerate(vector_only):
        print(f'{i+1}. [sim {doc["similarity"]:.3f}] {doc["content"][:80]}...')
    
    print('\n=== Vector + Rerank Top 3 ===')
    for i, doc in enumerate(reranked):
        print(f'{i+1}. [rerank {doc["rerank_score"]:.3f}] {doc["content"][:80]}...')
    
    # 순서 변화 있는지 체크
    v_ids = [d['id'] for d in vector_only]
    r_ids = [d['id'] for d in reranked]
    overlap = set(v_ids) & set(r_ids)
    print(f'\n📊 공통 문서: {len(overlap)}/3 — Reranker가 {3-len(overlap)}개 교체')

compare_search_quality('FOMC 점도표에서 2026년 전망')

## 🎯 과제

1. **테스트 세트 구축** — 본인 PDF에 대한 질문 10개 + 정답 페이지 정답지 작성
2. **정답률 측정** — Vector만 Top-3 안에 정답 포함률 vs Rerank 적용 후 포함률 비교
3. **카테고리 필터** — W5 과제에서 만든 filtered RPC 활용 → "FOMC 카테고리만 검색" 옵션
4. **중복 페이지 제거** — 같은 파일의 같은 페이지가 Top 3에 2번 나오면 1번으로 통합
5. **Claude 환각 테스트** — 지식베이스에 없는 질문을 일부러 물어보고 "정보 없음" 답변 검증

## 🔄 n8n vs Python 비교

| 개념 | n8n | Python |
|---|---|---|
| PDF 추출 | Extract From File 노드 | `pypdf.PdfReader` |
| 청킹 | Recursive Character Splitter sub-node | `langchain_text_splitters` |
| 메타데이터 생성 | Code 노드 | 딕셔너리 구성 |
| 배치 임베딩 | Default Data Loader | `openai_client.embeddings.create(input=list)` |
| Vector 검색 | Retrieve (20) | `supabase.rpc('match_documents', top=20)` |
| Rerank | Reranker Cohere sub-node | `cohere_client.rerank()` |
| 답변 + 인용 | AI Agent + Prompt | `claude.messages.create` + context |

**핵심 배움:** Hybrid RAG의 핵심은 **두 종류의 모델을 역할 분담**하는 것입니다.
- **Bi-encoder (OpenAI Embedding)**: 빠르지만 부정확 → 넓은 그물용
- **Cross-encoder (Cohere Rerank)**: 느리지만 정확 → 정밀 필터

이 구조가 있어야 정답률이 40%에서 88%로 올라갑니다.